# Paper #26 Implementation: Radio Emission from Solar Flares / 태양 플레어의 전파 방출
**Bastian, Benz, Gary 1998, ARA&A 36, 131-188**

**English.** This notebook reproduces the core radio-emission diagnostics discussed in the review. We compute: (1) plasma and cyclotron frequencies vs. coronal conditions, (2) gyrosynchrotron spectra (self-absorbed + optically thin) for a power-law electron distribution, (3) thermal bremsstrahlung vs. nonthermal gyrosynchrotron comparison, (4) type III drift rate and implied beam speed, (5) Razin suppression at low frequencies, and (6) a synthetic dynamic spectrum (f vs t) of a type III burst.

**Korean / 한국어.** 이 노트북은 리뷰에서 논의된 핵심 전파 방출 진단을 재현한다. (1) 코로나 조건에 따른 plasma·cyclotron 주파수, (2) power-law 전자 분포에 대한 gyrosynchrotron 스펙트럼(self-absorbed + 광학 얇음), (3) 열 bremsstrahlung vs 비열 gyrosynchrotron 비교, (4) type III drift rate와 유도된 빔 속도, (5) 저주파에서의 Razin 억제, (6) type III 버스트의 합성 dynamic spectrum (f vs t) 을 계산한다.

## 0. Setup / 준비

**English.** Load NumPy and Matplotlib. Define physical constants in CGS units.

**Korean / 한국어.** NumPy와 Matplotlib 로드. CGS 단위의 물리 상수 정의.

In [ ]:
"""Imports and physical constants for solar radio diagnostics."""
import numpy as np
import matplotlib.pyplot as plt

# CGS physical constants
E_CHARGE = 4.8032e-10    # statcoul
M_ELECTRON = 9.1094e-28  # g
C_LIGHT = 2.9979e10      # cm/s
K_BOLTZ = 1.3807e-16     # erg/K

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11

## 1. Plasma & Cyclotron Frequencies / Plasma·Cyclotron 주파수

**English.** The two foundational frequencies of solar radio physics:
$$\nu_{pe} = \sqrt{\frac{n_e e^2}{\pi m_e}} \approx 8980\sqrt{n_e}\;\text{Hz}, \qquad \nu_{Be} = \frac{eB}{2\pi m_e c} \approx 2.8\,B\;\text{MHz}.$$

Plasma frequency scales with $\sqrt{n_e}$; cyclotron frequency scales linearly with B.

**Korean / 한국어.** 태양 전파 물리의 두 기본 주파수. Plasma 주파수는 $\sqrt{n_e}$에 비례; cyclotron 주파수는 B에 비례.

In [ ]:
"""Compute plasma and electron cyclotron frequencies over coronal ranges."""


def plasma_frequency(n_e):
    """Return electron plasma frequency in Hz for density n_e [cm^-3].

    Uses nu_pe = sqrt(n_e e^2 / (pi m_e)).
    """
    return np.sqrt(n_e * E_CHARGE**2 / (np.pi * M_ELECTRON))


def cyclotron_frequency(B):
    """Return electron cyclotron frequency in Hz for B [Gauss]."""
    return E_CHARGE * B / (2 * np.pi * M_ELECTRON * C_LIGHT)


# Coronal density and field ranges
n_e_arr = np.logspace(8, 11, 100)
B_arr = np.logspace(0, 4, 100)

nu_pe = plasma_frequency(n_e_arr)
nu_Be = cyclotron_frequency(B_arr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.loglog(n_e_arr, nu_pe / 1e6, color="C0", lw=2)
ax1.axhspan(100, 3000, color="C0", alpha=0.1,
            label="nu_pe ~ 100-3000 MHz")
ax1.set_xlabel("electron density n_e (cm^-3)")
ax1.set_ylabel("plasma frequency nu_pe (MHz)")
ax1.set_title("Plasma frequency vs coronal density")
ax1.grid(True, which="both", alpha=0.3)
ax1.legend()

ax2.loglog(B_arr, nu_Be / 1e6, color="C3", lw=2)
ax2.axhspan(300, 3000, color="C3", alpha=0.1,
            label="nu_Be ~ 300-3000 MHz (B=100-1000 G)")
ax2.set_xlabel("magnetic field B (G)")
ax2.set_ylabel("cyclotron frequency nu_Be (MHz)")
ax2.set_title("Electron cyclotron frequency vs B")
ax2.grid(True, which="both", alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"n_e=1e9  -> nu_pe = {plasma_frequency(1e9)/1e6:.1f} MHz")
print(f"n_e=1e10 -> nu_pe = {plasma_frequency(1e10)/1e6:.1f} MHz")
print(f"B=100 G  -> nu_Be = {cyclotron_frequency(100)/1e6:.1f} MHz")
print(f"B=1000 G -> nu_Be = {cyclotron_frequency(1000)/1e6:.1f} MHz")

## 2. Gyrosynchrotron Spectrum (Power-Law Electrons) / Gyrosynchrotron 스펙트럼

**English.** Following Dulk & Marsh (1982), for a power-law electron distribution $n(E) \propto E^{-\delta}$, the emissivity and absorption are approximated by closed-form expressions. Full transfer is $T_B = T_{eff}(1-e^{-\tau})$, producing the universal inverted-U spectrum with peak at $\nu_{pk}$ where $\tau \approx 1$.

**Korean / 한국어.** Dulk & Marsh(1982)에 따라 power-law 전자 분포 $n(E) \propto E^{-\delta}$에서 emissivity와 흡수를 닫힌 형태로 근사. 전달해 $T_B = T_{eff}(1-e^{-\tau})$가 τ ≈ 1에서 peak하는 보편적 inverted-U 스펙트럼을 만든다.

In [ ]:
"""Gyrosynchrotron spectrum using Dulk-Marsh (1982) approximations."""


def gyrosynchrotron_Tb(nu_Hz, B, theta_deg, n_rel, path_length, delta):
    """Compute gyrosynchrotron brightness temperature T_B(nu).

    Uses Dulk & Marsh (1982) simplified expressions.

    Args:
        nu_Hz: Observing frequency array in Hz.
        B: Magnetic field in Gauss.
        theta_deg: Angle between line of sight and B in degrees.
        n_rel: Non-thermal electron density in cm^-3.
        path_length: Line of sight path in cm.
        delta: Power-law spectral index of electrons.

    Returns:
        Tuple of (T_B, tau, T_eff) arrays.
    """
    theta = np.deg2rad(theta_deg)
    sin_t = np.sin(theta)
    nu_B = cyclotron_frequency(B)
    s = nu_Hz / nu_B

    eta_coef = 3.3e-24 * 10**(-0.52 * delta)
    eta = (eta_coef * n_rel * B
           * sin_t**(-0.43 + 0.65 * delta)
           * s**(1.22 - 0.90 * delta))

    kappa_coef = 1.4e-9 * 10**(-0.22 * delta)
    kappa = (kappa_coef * n_rel / B
             * sin_t**(-0.09 + 0.72 * delta)
             * s**(-1.30 - 0.98 * delta))

    tau = kappa * path_length
    T_eff = eta / (kappa + 1e-60) * C_LIGHT**2 / (2 * K_BOLTZ * nu_Hz**2)
    T_b = T_eff * (1.0 - np.exp(-tau))
    return T_b, tau, T_eff


nu = np.logspace(np.log10(1e9), np.log10(5e10), 300)
B_loop = 500.0
theta_deg = 60.0
n_rel = 5e6
path_len = 1e9

deltas = [2, 3, 4, 5, 7]

fig, ax = plt.subplots(figsize=(9, 5.5))
for d in deltas:
    Tb, tau, Teff = gyrosynchrotron_Tb(nu, B_loop, theta_deg, n_rel, path_len, d)
    ax.loglog(nu / 1e9, Tb, label=f"delta = {d}")

ax.set_xlabel("frequency (GHz)")
ax.set_ylabel("brightness temperature T_B (K)")
ax.set_title(f"Gyrosynchrotron spectrum (B={B_loop:.0f} G, theta={theta_deg:.0f} deg)")
ax.legend(title="electron index")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

Tb4, _, _ = gyrosynchrotron_Tb(nu, B_loop, theta_deg, n_rel, path_len, 4)
nu_pk = nu[np.argmax(Tb4)]
print(f"Peak frequency for delta=4: {nu_pk/1e9:.2f} GHz")
print(f"Peak T_B: {Tb4.max():.2e} K")

## 3. Thermal Bremsstrahlung vs. Non-thermal Gyrosynchrotron / 열 bremsstrahlung vs 비열 GS

**English.** Thermal free-free optical depth $\tau_{ff} \propto n_e^2 L / (T^{3/2}\nu^2)$. Optically thick at low nu (T_B = T), optically thin at high nu (T_B decreases as nu^-2). Non-thermal GS dominates above ~3-5 GHz in active flares; free-free dominates below.

**Korean / 한국어.** 열 free-free 광학 두께 $\tau_{ff} \propto n_e^2 L / (T^{3/2}\nu^2)$. 저주파에서 광학 두꺼움(T_B = T), 고주파에서 광학 얇음(T_B ~ nu^-2). 비열 GS는 ~3-5 GHz 이상에서 지배, free-free는 그 이하에서 지배.

In [ ]:
"""Compare thermal free-free to non-thermal gyrosynchrotron."""


def thermal_freefree_tau(nu_Hz, n_e, T, L):
    """Thermal free-free optical depth (Dulk 1985, simplified)."""
    return 0.2 * n_e**2 * L / (T**1.5 * nu_Hz**2)


def freefree_Tb(nu_Hz, n_e, T, L):
    """Brightness temperature of thermal free-free source."""
    tau = thermal_freefree_tau(nu_Hz, n_e, T, L)
    return T * (1.0 - np.exp(-tau))


nu = np.logspace(np.log10(3e8), np.log10(5e10), 400)

T_hot = 2e7
n_e_hot = 1e10
L_hot = 2e9

Tb_ff = freefree_Tb(nu, n_e_hot, T_hot, L_hot)
Tb_gs, _, _ = gyrosynchrotron_Tb(nu, B=500, theta_deg=60, n_rel=5e6,
                                 path_length=1e9, delta=4)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.loglog(nu / 1e9, Tb_ff, label="thermal free-free (T=2e7 K, n_e=1e10)",
          color="C1", lw=2)
ax.loglog(nu / 1e9, Tb_gs, label="non-thermal GS (delta=4, B=500 G)",
          color="C0", lw=2)
ax.loglog(nu / 1e9, Tb_ff + Tb_gs, "--", color="k", label="combined")
ax.set_xlabel("frequency (GHz)")
ax.set_ylabel("brightness temperature T_B (K)")
ax.set_title("Thermal free-free vs non-thermal gyrosynchrotron")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Type III Drift Rate and Beam Speed / Type III drift rate와 빔 속도

**English.** Type III plasma emission frequency is $\nu \approx \nu_{pe}(n_e(h))$. For a hydrostatic atmosphere of scale height H, $df/dt = -(1/2)\nu\,v_{beam}/H$. Combined with a density model we recover $v_{beam}$.

**Korean / 한국어.** Type III plasma 방출 주파수 $\nu \approx \nu_{pe}(n_e(h))$. 정수 대기(scale height H)에서 $df/dt = -(1/2)\nu\,v_{beam}/H$. 밀도 모델과 결합하면 $v_{beam}$ 복원.

In [ ]:
"""Type III drift rate vs frequency for different beam speeds."""


def drift_rate(nu_Hz, v_beam, H=1e9):
    """Return df/dt (Hz/s) assuming nu = nu_pe on hydrostatic profile.

    df/dt = -(1/2) * nu * (v_beam / H). Negative sign = upward beam.
    """
    return -0.5 * nu_Hz * v_beam / H


nu_arr = np.logspace(7, 9, 100)
beam_speeds = [0.1, 0.2, 0.3, 0.5]

fig, ax = plt.subplots(figsize=(9, 5.5))
for v in beam_speeds:
    dfdt = np.abs(drift_rate(nu_arr, v * C_LIGHT))
    ax.loglog(nu_arr / 1e6, dfdt / 1e6, label=f"v = {v}c", lw=2)

dfdt_emp = 0.01 * (nu_arr / 1e6)**1.84 * 1e6
ax.loglog(nu_arr / 1e6, dfdt_emp / 1e6, "k--",
          label="Alvarez & Haddock 1973 empirical")

ax.set_xlabel("frequency nu (MHz)")
ax.set_ylabel("|df/dt| (MHz/s)")
ax.set_title("Type III drift rate vs frequency")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

nu_obs = 1e8
dfdt_obs = 225e6
H_assumed = 1e9
v_inferred = 2 * dfdt_obs * H_assumed / nu_obs
print(f"Observed drift 225 MHz/s at 100 MHz -> v_beam = {v_inferred/C_LIGHT:.2f} c")

## 5. Razin Suppression / Razin 억제

**English.** Below the Razin frequency $\nu_R = 20\,n_e/B$ Hz the ambient plasma refractive index n<1 suppresses relativistic beaming and cuts off gyrosynchrotron emission. High n_e/B ratio produces unusually steep low-frequency GS slopes.

**Korean / 한국어.** Razin 주파수 $\nu_R = 20\,n_e/B$ Hz 이하에서 주변 플라즈마의 굴절률 n<1이 상대론적 beaming을 억제하여 gyrosynchrotron 방출을 차단. 높은 n_e/B 비가 비정상적으로 가파른 저주파 GS 기울기를 만든다.

In [ ]:
"""Razin suppression of GS spectrum for variable ambient density."""


def razin_frequency(n_e, B, theta_deg=60):
    """Razin-Tsytovich frequency nu_R = 2 nu_pe^2 / (3 nu_Be sin theta)."""
    theta = np.deg2rad(theta_deg)
    nu_p = plasma_frequency(n_e)
    nu_B = cyclotron_frequency(B)
    return 2 * nu_p**2 / (3 * nu_B * np.sin(theta))


def razin_suppression(nu_Hz, n_e, B, theta_deg=60):
    """Phenomenological suppression factor exp(-nu_R/nu) (Belkora 1997)."""
    nu_R = razin_frequency(n_e, B, theta_deg)
    return np.exp(-nu_R / nu_Hz)


nu = np.logspace(9, 11, 300)
B_val = 500
theta = 60

n_e_list = [1e9, 3e10, 1e11, 3e11]
fig, ax = plt.subplots(figsize=(9, 5.5))

for n_e in n_e_list:
    Tb_gs, _, _ = gyrosynchrotron_Tb(nu, B_val, theta, n_rel=5e6,
                                     path_length=1e9, delta=4)
    sup = razin_suppression(nu, n_e, B_val, theta)
    Tb_razin = Tb_gs * sup
    nu_R = razin_frequency(n_e, B_val, theta)
    ax.loglog(nu / 1e9, Tb_razin,
              label=f"n_e={n_e:.0e}, nu_R={nu_R/1e9:.2f} GHz")

ax.set_xlabel("frequency (GHz)")
ax.set_ylabel("T_B (K) with Razin suppression")
ax.set_title(f"Razin suppression of GS spectrum (B={B_val} G)")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

for n_e in n_e_list:
    print(f"n_e={n_e:.1e}: nu_R = {razin_frequency(n_e, B_val)/1e9:.2f} GHz")

## 6. Synthetic Dynamic Spectrum of Type III Bursts / Type III 합성 dynamic spectrum

**English.** A dynamic spectrum displays flux vs. (frequency, time). We synthesize a group of type III bursts: each drifts $f(t) = f_0 \exp(-t/\tau_{drift})$ with a Gaussian temporal envelope. The superposition on the (f,t) plane reproduces Figure 8 of Bastian et al. (1998).

**Korean / 한국어.** Dynamic spectrum은 flux를 (주파수, 시간) 평면에 표시한다. 각 type III burst는 $f(t) = f_0 \exp(-t/\tau_{drift})$로 drift하며 Gaussian 시간 포락선을 가진다. (f,t) 평면상의 중첩이 Bastian et al.(1998) Figure 8의 형태를 재현.

In [ ]:
"""Synthetic dynamic spectrum of a group of type III bursts."""


def single_typeIII(t_grid, f_grid, t0, f0, tau_drift, tau_env, amp=1.0):
    """Generate a single type III burst on the (f, t) plane.

    Frequency drifts as f(t) = f0 * exp(-(t-t0) / tau_drift) with
    Gaussian envelope of width tau_env in time.

    Args:
        t_grid: Time axis in seconds.
        f_grid: Frequency axis in MHz.
        t0: Burst start time in seconds.
        f0: Initial (high) frequency in MHz.
        tau_drift: Drift e-folding time in seconds.
        tau_env: Gaussian temporal width in seconds.
        amp: Peak amplitude.

    Returns:
        2D array of flux values with shape (len(f_grid), len(t_grid)).
    """
    T, F = np.meshgrid(t_grid, f_grid, indexing="xy")
    t_rel = T - t0
    f_burst = f0 * np.exp(-t_rel / tau_drift)
    sig_f = 0.08 * f_burst
    spatial = np.exp(-0.5 * ((F - f_burst) / sig_f)**2)
    temporal = np.exp(-0.5 * (t_rel / tau_env)**2)
    profile = amp * spatial * temporal
    profile[t_rel < -2 * tau_env] = 0
    return profile


t_grid = np.linspace(0, 40, 800)
f_grid = np.linspace(100, 600, 400)

rng = np.random.default_rng(42)
spec = np.zeros((len(f_grid), len(t_grid)))

for i in range(8):
    t0 = rng.uniform(4, 34)
    f0 = rng.uniform(400, 600)
    tau_d = rng.uniform(1.5, 3.0)
    tau_e = rng.uniform(0.6, 1.4)
    amp = rng.uniform(0.6, 1.0)
    spec += single_typeIII(t_grid, f_grid, t0, f0, tau_d, tau_e, amp)

spec += 0.03 * rng.standard_normal(spec.shape)
spec = np.clip(spec, 0, None)

fig, ax = plt.subplots(figsize=(10, 5.5))
im = ax.imshow(spec, aspect="auto", origin="lower",
               extent=(t_grid[0], t_grid[-1], f_grid[0], f_grid[-1]),
               cmap="inferno", interpolation="bilinear")
ax.set_xlabel("time (s)")
ax.set_ylabel("frequency (MHz)")
ax.set_title("Synthetic type III burst group (dynamic spectrum)")
plt.colorbar(im, ax=ax, label="relative flux")
plt.tight_layout()
plt.show()

## Summary / 요약

**English.** We reproduced the foundational radio diagnostics of Bastian, Benz, Gary (1998). (1) Coronal n_e ~ 1e9-1e10 gives nu_pe ~ 280-900 MHz; B ~ 100-1000 G gives nu_Be ~ 280-2800 MHz, the two axes organizing flare radio physics. (2) The Dulk-Marsh (1982) GS spectrum produces the universal inverted-U with nu_pk in 5-15 GHz, matching observations. (3) Thermal free-free dominates below ~3 GHz while non-thermal GS dominates above. (4) Type III drift rates of ~225 MHz/s at 100 MHz imply v_beam ~ 0.15-0.3 c. (5) Razin suppression at nu_R = 20 n_e/B Hz explains steep low-frequency slopes in high-density flares. (6) The synthetic dynamic spectrum demonstrates how decimetric type III bursts appear as drifting stripes on the (f,t) plane.

**Korean / 한국어.** 우리는 Bastian, Benz, Gary(1998)의 기초 전파 진단을 재현했다. (1) 코로나 n_e ~ 1e9-1e10은 nu_pe ~ 280-900 MHz를 주고, B ~ 100-1000 G는 nu_Be ~ 280-2800 MHz를 준다 - 플레어 전파 물리의 두 축. (2) Dulk-Marsh(1982) GS 스펙트럼은 nu_pk ~ 5-15 GHz의 보편적 inverted-U를 만들며 관측과 일치. (3) ~3 GHz 이하는 열 free-free가, 이상은 비열 GS가 지배. (4) 100 MHz에서 ~225 MHz/s의 Type III drift rate는 v_beam ~ 0.15-0.3 c를 의미. (5) nu_R = 20 n_e/B Hz의 Razin 억제가 고밀도 플레어의 가파른 저주파 기울기를 설명. (6) 합성 dynamic spectrum은 decimetric type III가 (f,t) 평면의 drift stripe로 나타남을 보인다.